# Notebook 6: Feature Space Analysis

Go beyond accuracy numbers. Visualize **what the model represents internally**
by comparing:

1. **Weight drift** — how much did each layer's weights change from ImageNet?
2. **t-SNE of embeddings** — are flower classes separable in feature space?
3. **Cosine similarity** — how similar are the pretrained vs fine-tuned features?

This tells you exactly WHERE transfer learning adapts and WHERE it keeps
the original features.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from torchvision import models
from src.data import get_dataloaders
from src.model import create_model
from tqdm import tqdm

DEVICE = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Weight Drift — How Much Did Each Layer Change?

In [ ]:
# Load original ImageNet weights and fine-tuned weights
original = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

fine_tuned = create_model(num_classes=102, mode='fine_tune', unfreeze_from='layer3')
fine_tuned.load_state_dict(torch.load('../models/fine_tune_best.pth', map_location='cpu'))

# Compare weights layer by layer
drift_per_layer = {}
for name_orig, param_orig in original.named_parameters():
    # Skip the fc layer (it's completely different)
    if 'fc' in name_orig:
        continue

    # Find matching parameter in fine-tuned model
    param_ft = dict(fine_tuned.named_parameters()).get(name_orig)
    if param_ft is not None:
        # Relative change: how much did this tensor move?
        diff = (param_ft.data - param_orig.data).norm()
        orig_norm = param_orig.data.norm()
        relative_drift = (diff / (orig_norm + 1e-8)).item()

        # Group by layer block
        block = name_orig.split('.')[0]  # e.g., 'layer1', 'layer2', etc.
        if block not in drift_per_layer:
            drift_per_layer[block] = []
        drift_per_layer[block].append(relative_drift)

# Average drift per block
blocks = list(drift_per_layer.keys())
avg_drift = [np.mean(drift_per_layer[b]) for b in blocks]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71' if d < 0.01 else '#f39c12' if d < 0.1 else '#e74c3c' for d in avg_drift]
bars = ax.bar(blocks, avg_drift, color=colors)
ax.set_ylabel('Relative Weight Drift (||Δw|| / ||w||)')
ax.set_title('How Much Did Each Layer Change From ImageNet?\n'
             '(green = barely moved, red = significantly changed)')
ax.grid(True, alpha=0.3, axis='y')

for bar, d in zip(bars, avg_drift):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{d:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../results/weight_drift.png', dpi=150)
plt.show()

print("\nINTERPRETATION:")
print("- Early layers (conv1, bn1, layer1, layer2) should show near-zero drift")
print("  → They were frozen OR their features are already universal")
print("- Later layers (layer3, layer4) should show larger drift")
print("  → They adapted their high-level features to recognize flowers")

## 2. t-SNE — Are Flower Classes Separable in Feature Space?

Extract the 512-dimensional feature vector (just before the classification head)
for each image, then project to 2D with t-SNE. If clusters form by class,
the backbone learned meaningful representations.

In [ ]:
@torch.no_grad()
def extract_features(model, data_loader, device):
    """Extract the 512-dim feature vector before the classification head."""
    model.eval()
    features = []
    labels = []

    # Hook into avgpool to get features before fc
    hook_output = []
    def hook(module, input, output):
        hook_output.append(output.squeeze().cpu())

    handle = model.avgpool.register_forward_hook(hook)

    for images, lbls in tqdm(data_loader, desc='Extracting features'):
        images = images.to(device)
        _ = model(images)
        features.append(hook_output[-1])
        labels.append(lbls)

    handle.remove()
    return torch.cat(features).numpy(), torch.cat(labels).numpy()


# Extract features from BOTH models
_, val_loader, _ = get_dataloaders(data_dir='../data', batch_size=32)

# Pretrained (no fine-tuning) — feature extraction model
model_pretrained = create_model(num_classes=102, mode='feature_extract').to(DEVICE)
feats_pretrained, labels_p = extract_features(model_pretrained, val_loader, DEVICE)

# Fine-tuned model
fine_tuned = fine_tuned.to(DEVICE)
feats_finetuned, labels_f = extract_features(fine_tuned, val_loader, DEVICE)

print(f'Feature shape: {feats_pretrained.shape}')

In [ ]:
# t-SNE projection
# Use a subset of classes for cleaner visualization
show_classes = list(range(0, 102, 10))  # Every 10th class
mask_p = np.isin(labels_p, show_classes)
mask_f = np.isin(labels_f, show_classes)

tsne = TSNE(n_components=2, random_state=42, perplexity=30)

proj_pretrained = tsne.fit_transform(feats_pretrained[mask_p])
proj_finetuned = tsne.fit_transform(feats_finetuned[mask_f])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

scatter1 = axes[0].scatter(proj_pretrained[:, 0], proj_pretrained[:, 1],
                           c=labels_p[mask_p], cmap='tab20', s=30, alpha=0.7)
axes[0].set_title('Pretrained (ImageNet features, no fine-tuning)\n'
                  'Clusters = features already somewhat distinguish flowers')
axes[0].set_xticks([]); axes[0].set_yticks([])

scatter2 = axes[1].scatter(proj_finetuned[:, 0], proj_finetuned[:, 1],
                           c=labels_f[mask_f], cmap='tab20', s=30, alpha=0.7)
axes[1].set_title('Fine-tuned (adapted to flowers)\n'
                  'Tighter clusters = better class separation')
axes[1].set_xticks([]); axes[1].set_yticks([])

plt.suptitle('t-SNE: Feature Space Before vs After Fine-Tuning', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/tsne_comparison.png', dpi=150)
plt.show()

print("\nWHAT TO LOOK FOR:")
print("- Left (pretrained): some structure, but classes overlap")
print("- Right (fine-tuned): tighter, more separated clusters")
print("- This is literally what fine-tuning does: reshapes the feature space")
print("  so that similar flowers are close together, different ones are far apart")

## 3. Cosine Similarity: Pretrained vs Fine-Tuned Features

For each image, compute how similar the pretrained and fine-tuned feature
vectors are. High similarity = that image didn't need much adaptation.

In [ ]:
from torch.nn.functional import cosine_similarity

cos_sim = cosine_similarity(
    torch.tensor(feats_pretrained),
    torch.tensor(feats_finetuned),
    dim=1
).numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(cos_sim, bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(cos_sim.mean(), color='red', linestyle='--', label=f'Mean: {cos_sim.mean():.3f}')
axes[0].set_xlabel('Cosine Similarity')
axes[0].set_ylabel('Number of Images')
axes[0].set_title('Feature Similarity: Pretrained vs Fine-Tuned')
axes[0].legend()

# Per-class average similarity
class_sim = {}
for cls in np.unique(labels_p):
    mask = labels_p == cls
    class_sim[cls] = cos_sim[mask].mean()

sorted_classes = sorted(class_sim, key=class_sim.get)
bottom_10 = sorted_classes[:10]  # Most changed
top_10 = sorted_classes[-10:]    # Least changed

show = bottom_10 + top_10
sims = [class_sim[c] for c in show]
colors = ['#e74c3c']*10 + ['#2ecc71']*10

axes[1].barh(range(len(show)), sims, color=colors)
axes[1].set_yticks(range(len(show)))
axes[1].set_yticklabels([f'Class {c}' for c in show])
axes[1].set_xlabel('Cosine Similarity')
axes[1].set_title('Most Changed (red) vs Least Changed (green) Classes')
axes[1].axvline(cos_sim.mean(), color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../results/cosine_similarity.png', dpi=150)
plt.show()

print("\nINTERPRETATION:")
print("- High similarity → ImageNet features were already good for this flower")
print("  (maybe it looks like an ImageNet object?)")
print("- Low similarity → fine-tuning had to significantly reshape the representation")
print("  (this flower has very unique visual features)")